# 강의 06 · 실습 3 — 패턴 3 핸드오프 · (1) 강사 시연

## 1. 문제상황

- 온라인 가구 쇼핑몰 고객센터는 주문 접수 담당과 배송 담당이 나뉘어 있습니다.
- 고객이 주문 접수 담당에게 「어제 주문한 의자가 지금 어디까지 왔나요」라고 물으면, 접수 담당은 「배송팀에 물어보세요」라고 답하고 끝냅니다.
- 고객은 전화를 끊고 배송 담당에게 다시 전화해 같은 이야기를 처음부터 합니다.
- 접수 담당이 배송 담당에게 문의를 넘길 수 있다면 고객은 한 번의 문의로 답을 받을 수 있습니다.

## 2. 문제와 목표

- **문제**: 접수 에이전트가 자기 범위 밖의 문의를 받으면 안내만 하고 끝나므로, 고객이 에이전트를 바꿔 다시 문의해야 합니다. 대화 이력도 이어지지 않습니다.
- **목표**: 접수 에이전트가 받은 문의에 배송 이야기가 나오면 핸드오프 도구로 에이전트를 배송으로 바꾸고, 같은 대화 이력을 이어받은 배송 에이전트가 같은 대화에서 답하는 처리 흐름을 만듭니다.
    - 에이전트 둘: 접수(주문 번호·품목 문의), 배송(배송 상태 안내). 노드는 agent 하나이고 `active_agent` 값으로 지침과 도구를 고릅니다.
    - 핸드오프 도구: 접수 에이전트가 부르면 `Command`로 `active_agent`를 배송으로 바꿉니다.
    - 같은 대화 이력: 상태의 `messages` 키(`add_messages` 리듀서).
- **목표 달성 여부의 판정 기준**: 배송 문의를 접수 에이전트에게 넣었을 때, 접수 담당이 한 번 응대하며 배송 담당에게 넘기고, 같은 대화 이력 위에서 배송 담당이 답하는 것을 실행 결과에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec06_ex03_s1_diagram.svg)

## 4. 단계별 요구사항

1. **상태를 정의합니다.**
    - 대화 이력(`messages`)과 현재 에이전트(`active_agent`) 키 두 개를 가지는 상태를 선언합니다.
    - `messages` 키에는 `add_messages` 리듀서를 걸어 노드가 돌려준 메시지가 덮이지 않고 이어 붙도록 합니다.
2. **핸드오프 도구를 만듭니다.**
    - to_delivery 도구는 문자열이 아니라 `Command`를 돌려줍니다.
    - `Command`의 `update`에 「배송 에이전트에게 넘겼습니다」라는 `ToolMessage`와 `active_agent`를 배송으로 바꾸는 값을 담습니다.
    - 도구는 `ToolRuntime`에서 자기 호출 id를 받아 `ToolMessage`에 붙입니다.
3. **에이전트별 지침과 agent 노드를 만듭니다.**
    - 접수 에이전트의 시스템 프롬프트는 주문 번호·품목만 다루고 배송 이야기가 나오면 도구로 넘기라고 지시하며, 도구 목록은 to_delivery 하나입니다.
    - 배송 에이전트의 시스템 프롬프트는 배송 상태를 두 문장으로 안내하라고 지시하며, 도구 목록은 비어 있습니다.
    - agent 노드는 `active_agent` 값으로 지침과 도구 목록을 고른 뒤, 도구가 있으면 `bind_tools`로 묶은 모델을, 없으면 그대로의 모델을 불러 답을 `messages`에 덧붙입니다.
4. **그래프에 노드를 등록합니다.**
    - agent 노드와, 핸드오프 도구를 실행하는 tools 노드(`ToolNode`) 두 개를 등록합니다.
5. **엣지를 연결합니다.**
    - START에서 agent로 가는 고정 엣지를 추가합니다.
    - agent 뒤에는 마지막 메시지에 도구 호출이 실려 있으면 tools로, 없으면 END로 가는 조건부 엣지를 추가합니다.
    - tools 뒤에는 agent로 돌아가는 고정 엣지를 추가합니다.
6. **그래프를 컴파일하고 실행합니다.**
    - 배송 문의를 대화 이력에 넣고 시작 에이전트를 접수로 두어 실행한 뒤, 최종 에이전트 값과 메시지 수와 마지막 답을 출력합니다.

## 5. 코드 골격 — LangGraph 5단

랭그래프(LangGraph)로 그래프를 세우는 순서는 다음 다섯 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 다섯 단계와 하나씩 대응합니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 상태 정의 | 대화 이력과 현재 에이전트의 키를 선언합니다 | `class HandoffState(TypedDict)`, `Annotated[list, add_messages]` | 1 |
| ② 노드 함수 정의 | 핸드오프 도구와, 에이전트에 따라 달라지는 agent 함수를 만듭니다 | `@tool`, `Command(update=...)`, `def agent(state) -> dict` | 2, 3 |
| ③ 그래프 빌더 생성과 노드 등록 | 빈 그래프를 열고 agent와 도구 실행 노드를 등록합니다 | `StateGraph(HandoffState)`, `add_node`, `ToolNode` | 4 |
| ④ 엣지 연결 | 도구 호출 여부로 나뉘고 도구에서 agent로 돌아오는 분기를 지정합니다 | `add_edge`, `add_conditional_edges` | 5 |
| ⑤ 컴파일과 실행 | 연결을 확정하고 대화 이력과 시작 에이전트를 넣어 실행합니다 | `compile()`, `invoke()` | 6 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv
from typing import Annotated, TypedDict

from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage, ToolMessage
from langchain.tools import ToolRuntime
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.types import Command

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
print("모델 준비를 마쳤습니다.")

### 단계 ① — 상태 정의 (요구사항 1)

그래프가 도는 동안 모든 노드가 함께 읽고 쓰는 키를 선언합니다. `messages`는 대화 이력이며 `add_messages` 리듀서가 새 메시지를 뒤에 이어 붙입니다. `active_agent`는 지금 누가 답하는지를 들고 있는 상태 변수이며, 이 한 키의 값이 agent 노드의 행동을 바꿉니다.

In [ ]:
class HandoffState(TypedDict):
    messages: Annotated[list, add_messages]   # 대화 이력 (리듀서가 뒤에 이어 붙인다)
    active_agent: str                          # 현재 에이전트 ("접수" 또는 "배송")


print("상태의 키:", list(HandoffState.__annotations__))

### 단계 ② — 노드 함수 정의 (요구사항 2, 3)

- 핸드오프 도구 `to_delivery`는 `@tool`을 붙인 파이썬 함수이며, 문자열 대신 `Command`를 돌려줍니다. `Command(update=...)`는 상태를 갱신하는 인자입니다. 도구가 상태의 `active_agent`를 배송으로 바꿔 넣습니다.
- `ToolRuntime` 인자는 모델이 이 도구를 호출한 id를 들고 있습니다. 도구가 돌려주는 `ToolMessage`에 그 id를 붙여야 모델이 어느 호출의 결과인지 압니다.
- 에이전트별 시스템 프롬프트와 도구 목록은 딕셔너리 두 개에 모아 둡니다. 에이전트가 바뀌면 무엇이 따라 바뀌는지가 이 두 딕셔너리에 전부 적혀 있습니다.
- agent 노드는 하나뿐입니다. 매 진입마다 `active_agent` 값으로 지침과 도구를 꺼내 모델을 부릅니다. 에이전트마다 노드를 따로 두지 않습니다.

In [ ]:
@tool
def to_delivery(runtime: ToolRuntime) -> Command:
    """배송 상태·송장 문의는 배송 에이전트에게 넘긴다."""
    print("    [tool] to_delivery -> active_agent 교체")
    return Command(update={
        "messages": [ToolMessage("배송 에이전트에게 넘겼습니다.", tool_call_id=runtime.tool_call_id)],
        "active_agent": "배송",
    })


PROMPTS = {
    "접수": ("주문 접수 에이전트다. 주문 번호·품목만 다룬다. "
             "배송 상태나 송장 이야기가 나오면 도구로 배송 에이전트에게 넘긴다."),
    "배송": "배송 에이전트다. 배송 상태를 두 문장으로 안내한다. 도구는 쓰지 않는다.",
}
AGENT_TOOLS = {"접수": [to_delivery], "배송": []}


def agent(state: HandoffState) -> dict:
    """현재 에이전트의 지침과 도구로 응대한다. 노드는 하나뿐이다."""
    who = state["active_agent"]
    tools = AGENT_TOOLS[who]
    model = llm.bind_tools(tools) if tools else llm
    res = model.invoke([SystemMessage(PROMPTS[who])] + state["messages"])
    print(f"  [agent] 진입 active_agent={who!r} tool_calls={len(res.tool_calls or [])}")
    return {"messages": [res]}

### 단계 ③ — 그래프 빌더 생성과 노드 등록 (요구사항 4)

`StateGraph`에 상태를 넘겨 빈 그래프를 열고, `add_node`로 함수마다 이름을 붙여 등록합니다. tools 노드는 함수 대신 `ToolNode`에 핸드오프 도구 목록을 넣어 등록합니다. `ToolNode`는 마지막 메시지에 실린 도구 호출을 찾아 도구를 실행하고, 도구가 돌려준 `Command`의 갱신을 상태에 적용합니다.

In [ ]:
g = StateGraph(HandoffState)
g.add_node("agent", agent)
g.add_node("tools", ToolNode([to_delivery]))

print("등록한 노드:", list(g.nodes))

### 단계 ④ — 엣지 연결 (요구사항 5)

`add_edge`는 고정된 순서로 연결합니다. `add_conditional_edges`는 판단 함수가 돌려준 이름으로 다음 노드가 나뉘는 분기를 추가합니다. 판단 함수 `need_tool`은 마지막 메시지에 도구 호출이 실려 있으면 tools, 없으면 END를 돌려줍니다. tools 뒤에는 agent로 돌아가는 고정 엣지를 추가합니다. 도구가 갱신한 에이전트 값을 다음 agent 진입이 읽으므로 에이전트를 바꾸는 전용 노드는 필요하지 않습니다.

In [ ]:
def need_tool(state: HandoffState) -> str:
    """마지막 메시지에 도구 호출이 실려 있으면 tools, 아니면 END."""
    last = state["messages"][-1]
    return "tools" if getattr(last, "tool_calls", None) else END


g.add_edge(START, "agent")
g.add_conditional_edges("agent", need_tool, ["tools", END])
g.add_edge("tools", "agent")   # 도구가 끝나면 agent로 돌아온다

print("고정 엣지 수:", len(g.edges))

### 단계 ⑤ — 컴파일과 실행 (요구사항 6)

`compile()`이 연결을 확정해 실행 가능한 그래프를 돌려줍니다. `invoke`에 대화 이력과 시작 에이전트를 넣으면 최종 상태가 돌아옵니다. 실행 중 agent 노드와 도구가 출력하는 진입 줄로 에이전트가 바뀌는 순간을 봅니다. 아래에서는 배송 문의를 접수 에이전트에게 넣습니다.

In [ ]:
graph = g.compile()

ASK = "어제 주문한 사무용 의자가 지금 어디까지 왔는지 알려주세요."

print(f"=== 문의: {ASK} ===")
out = graph.invoke({"messages": [HumanMessage(ASK)], "active_agent": "접수"})

print()
print(f"[최종 상태] active_agent={out['active_agent']!r} 메시지 수={len(out['messages'])}")
print("--- 마지막 답 ---")
print(out["messages"][-1].content)

## 7. 실행 결과 확인

위 실행 결과에서 다음 세 가지를 확인합니다.

1. `[agent] 진입` 줄이 두 번 출력됩니다. 첫 줄은 `active_agent='접수'`이고 `tool_calls=1`이며, 둘째 줄은 `active_agent='배송'`이고 `tool_calls=0`입니다. 같은 노드가 다른 에이전트로 두 번 돌았습니다.
2. 두 진입 줄 사이에 `[tool] to_delivery` 줄이 출력됩니다. 도구가 돌려준 `Command`가 `active_agent`를 바꾼 지점입니다.
3. 최종 상태의 `active_agent`는 배송이고, 메시지 수는 4입니다(문의, 접수 에이전트의 도구 호출, 도구 결과, 배송 에이전트의 답). 마지막 답은 배송 에이전트의 시스템 프롬프트대로 배송 상태 안내입니다.

agent 노드 하나가 에이전트 값에 따라 다른 지침과 도구로 두 번 돈 것이 핸드오프가 동작한 증거입니다.